<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# ROS 2 Nodes, Topics and Interfaces

The previous notebook introduced nodes, topics and messages as concepts. How can the graph reveal which nodes publish data, and how can an observer check whether their messages arrive?

A small text publisher makes both observations possible without a robot or a custom Python program. By the end of this notebook, the distinction between a **node**, a **topic** and a **message** will be visible in command output, together with the message types of the Duckiebot. The same example will also show why communicating components need an agreed message type.

## Before you start: the nodes need a router

ROS 2 nodes in this learning experience find each other through a **zenoh router** (notebook 04 explains why). Two situations are possible:

- **No robot yet.** Start a router inside the editor. Open a terminal (`Terminal -> New Terminal`) and run:

  ```bash
  ZENOH_CONFIG_OVERRIDE= ros2 run rmw_zenoh_cpp rmw_zenohd
  ```

  It prints `Started Zenoh router with id ...` and keeps running: leave this terminal alone for the rest of the notebooks and open new ones for the commands below. Every other terminal of the editor connects to this router automatically. (The empty `ZENOH_CONFIG_OVERRIDE=` in front of the command matters; notebook 04 explains what the variable does.)
- **A robot with its ROS 2 stack running** (`dts stack up -R ROBOTNAME ros2/duckiebot`, see the README). Point each terminal you open at the robot's router:

  ```bash
  export ZENOH_CONFIG_OVERRIDE='mode="client";connect/endpoints=["tcp/ROBOTNAME.local:7447"]'
  ```

  The robot's own nodes then show up in the listings below, next to the ones you start.

If a command fails with `Error opening session: Unable to connect to any of [tcp/...:7447]`, no router is reachable at that address: one of the two conditions above is not met.

## Is anything publishing yet?

The commands in this notebook run in two terminals. Terminal 1 will run the publisher; Terminal 2 will observe it.

Open Terminal 1 and, before starting a publisher, inspect the visible nodes:

```bash
ros2 node list
```

With no application nodes running (and no robot connected), the command prints nothing and returns to the prompt. An empty listing means that this observation discovered no visible nodes. If you are connected to a robot, you see its nodes instead, for example `/ROBOTNAME/camera_node`.

## Give the graph something to carry

A publisher needs a topic name, a message type, and values for the message fields. A simple text message is sufficient for this investigation.

The `ros2 interface show` command, introduced in notebook 01, reveals the field that must be filled:

```bash
ros2 interface show std_msgs/msg/String
```

Omitting the introductory comments, this shows:

```shell
string data
```

The message contains one field, `data`, holding text. The command-line tool can create a publisher and fill that field directly. In Terminal 1, start a publisher:

```bash
ros2 topic pub --rate 1 --node-name example_publisher /chatter std_msgs/msg/String "{data: '42'}"
```

Here `--rate 1` establishes the publishing frequency in hertz (one message per second) and `--node-name` gives the publishing node an identifiable name. `/chatter` is the topic, `std_msgs/msg/String` is its message type, and `"{data: '42'}"` assigns the text `42` to the `data` field (the argument is written in YAML).

The output should begin with:

```shell
publisher: beginning loop
publishing #1: std_msgs.msg.String(data='42')

publishing #2: std_msgs.msg.String(data='42')
```

The numbering continues as messages are published. This command keeps running, so the terminal does not return to its prompt. Leave it running while the second terminal inspects the graph.

## Find the node and its topic

Open Terminal 2 and repeat the node listing:

```bash
ros2 node list
```

The listing should now contain the node created by the publishing command:

```shell
/example_publisher
```

The next question is which topic carries its data. The `-t` option includes message types in the topic listing:

```bash
ros2 topic list -t
```

The listing should include:

```shell
/chatter [std_msgs/msg/String]
/parameter_events [rcl_interfaces/msg/ParameterEvent]
/rosout [rcl_interfaces/msg/Log]
```

The `/chatter` entry matches the topic and type selected in Terminal 1. The other two entries support parameter-change notifications and ROS logging; they are not additional data streams.

The distinction is now visible: `/example_publisher` names a participating node, while `/chatter` names a message stream. The node performs the publishing work; the topic identifies where subscribers can receive its messages.

**If a listing comes back empty** although the publisher is running: every `ros2` command opens a new session with the router, and it takes a few seconds for that session to learn the graph. Ask the tool to wait and observe directly, without its background helper (the "daemon"):

```bash
ros2 node list --no-daemon --spin-time 5
```

Keep this form in mind; it is the most reliable way to inspect a graph that has just changed.

## Check that a message actually arrives

The publisher is running, and the second terminal has discovered its node and topic. Receiving a message is the next check, because a visible interface alone does not establish delivery.

To "listen" to the data flowing in a topic, the `ros2 topic echo` command creates a temporary subscriber. The `--once` option requests one message and then exits. In Terminal 2, run:

```bash
ros2 topic echo --once /chatter
```

The received message should appear as:

```shell
data: '42'
---
```

The `data` field contains the text sent by Terminal 1. The `---` marks the end of the displayed message. After receiving it, the command returns to the prompt. To observe a continuous stream, omit `--once`; stop the continuous observation with `Ctrl-C`.

If the command remains silent, it is still waiting; that is different from the empty node listing before the publisher started. The first check is whether Terminal 1 is still publishing and whether both terminals use the same router (`printenv ZENOH_CONFIG_OVERRIDE` in both).

The publisher did not need to name this subscriber. It published on `/chatter`, and the temporary subscriber joined that topic. Other subscribers can join the same stream without changing the publisher.

## Why must the message types agree?

The displayed value resembles the number 42, but the published message contains text. Would a subscriber expecting an integer receive the same data?

The integer message definition makes the difference visible:

```bash
ros2 interface show std_msgs/msg/Int32
```

Omitting introductory comments, the definition contains:

```shell
int32 data
```

Both message definitions use the field name `data`, but one field contains text and the other a signed 32-bit integer. A subscriber expecting `Int32` does not receive the publisher's `String` messages merely because the topic and field names match. ROS 2 does not convert the text into an integer. Try it: `ros2 topic echo --once --timeout 5 /chatter std_msgs/msg/Int32` creates an `Int32` subscription on the topic, stays silent and returns after five seconds. Nothing arrives, and no error tells you why; a wrong type is one of the things to check when a subscriber is silent (notebook 10).

The meaning of a value also belongs to the interface agreement. Two components exchanging a speed can agree on a `float32` yet disagree about whether its value is in m/s or in wheel revolutions per second. A usable interface defines both the structure and the meaning of its data.

Delivery policies must also be compatible. ROS 2 calls these **quality of service (QoS)** settings; the defaults used by these commands support this exchange. Sensor drivers can use different policies, so a subscriber must use compatible settings; notebooks 07 and 10 look at them.

## The interfaces of the Duckiebot

The same tool documents the messages that flow on the robot. Three of them appear throughout this learning experience:

```bash
ros2 interface show duckietown_msgs/msg/Twist2DStamped
```

```shell
std_msgs/Header header
	builtin_interfaces/Time stamp
		int32 sec
		uint32 nanosec
	string frame_id
float32 v
float32 omega
```

A car command: a forward speed `v` (m/s) and an angular speed `omega` (rad/s, positive turns left), with a `header` carrying the time stamp. This is what your node will publish in notebook 09.

```bash
ros2 interface show duckietown_msgs/msg/WheelsCmdStamped
```

contains `float32 vel_left` and `float32 vel_right`: the speed of each wheel, produced by `kinematics_node` from a `Twist2DStamped`.

```bash
ros2 interface show duckietown_msgs/msg/WheelEncoderStamped
```

```shell
uint8 ENCODER_TYPE_ABSOLUTE=0
uint8 ENCODER_TYPE_INCREMENTAL=1
std_msgs/Header header
	...
int32 data
uint16 resolution
uint8 type
```

A wheel encoder reading: `data` counts ticks, `resolution` says how many ticks make one wheel revolution, and the two constants document the possible values of `type`.

The camera publishes a standard ROS 2 message, `sensor_msgs/msg/CompressedImage`: a `format` string (`jpeg`) and the raw bytes of the encoded image in `data`.

Displaying a definition establishes that the interface is installed and available to the terminal. Checking live delivery requires a running publisher and an observation of its messages, which is what notebook 08 does on the robot.

## Finish the observation

Stop the publisher with `Ctrl-C` in Terminal 1. Its publication output stops and the shell prompt returns.

In Terminal 2, repeat:

```bash
ros2 node list --no-daemon --spin-time 5
```

The listing should become empty again (or show only the robot's nodes) after discovery reflects the publisher's departure.

The investigation began with an installed message definition and no visible application nodes. Starting a publisher added a named node and a typed topic; receiving a message established delivery. Stopping the publisher removed that node from the graph.

These observations depended on both terminals discovering the same participants, through the same router. The next notebook looks at how that discovery works.

## Further reading

The ROS 2 Jazzy documentation describes [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html), [services](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Services.html), [actions](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Actions.html) and [parameters](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Parameters.html). The message definitions of the Duckietown stack are in the [`duckietown_msgs` package of `dt-ros2-commons`](https://github.com/duckietown/dt-ros2-commons/tree/ente/packages/duckietown_msgs/msg).

Let's move to [the next notebook about discovery and names](../notebooks/04_discovery_zenoh_and_names.ipynb).